# Módulo 03 · Búsqueda léxica con BM25

Se implementa el algoritmo de ranking clásico de los motores de búsqueda y se aplica a
los fragmentos de NovaTech.

In [1]:
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA_DIR = ROOT / "data" / "novatech"

## Tokenización para búsqueda

Antes de indexar, el texto se convierte en términos normalizados: minúsculas, sin
tildes, sin palabras vacías y con un *stemming* ligero que agrupa variantes.

In [2]:
from fundamenta.text import light_stem, tokenize

print(tokenize("¿Cuántos días de vacaciones me corresponden?", stem=False))
print(tokenize("¿Cuántos días de vacaciones me corresponden?"))
print()
for group in [("vacación", "vacaciones"), ("permiso", "permisos"), ("reembolsa", "reembolso")]:
    print(group, "->", {light_stem(tokenize(word, stem=False)[0]) for word in group})

['dias', 'vacaciones', 'corresponden']
['dia', 'vacacio', 'corresp']

('vacación', 'vacaciones') -> {'vacacio'}
('permiso', 'permisos') -> {'permis'}
('reembolsa', 'reembolso') -> {'reembol'}


> **Nota.** Esta tokenización es distinta de la que utiliza internamente un LLM. Un
> modelo de lenguaje divide el texto en subpalabras (mediante algoritmos como BPE) para
> procesarlo; aquí se busca que dos formas de una misma palabra coincidan al buscar.

## Frecuencia inversa de documento (IDF)

Un término que aparece en todos los fragmentos ("NovaTech") apenas ayuda a distinguir
unos de otros; uno poco frecuente ("mudanza") es muy informativo.

In [3]:
from fundamenta.bm25 import BM25Index
from fundamenta.chunking import chunk_documents, contextualize
from fundamenta.loaders import load_directory

chunks = chunk_documents(load_directory(DATA_DIR))
index = BM25Index().fit([contextualize(c) for c in chunks])

for word in ["NovaTech", "empresa", "vacaciones", "mudanza", "MacBook"]:
    term = tokenize(word)[0]
    print(f"{word:12} -> {term:9} IDF = {index.idf.get(term, 0):.2f}")

NovaTech     -> novatec   IDF = 0.35
empresa      -> empres    IDF = 1.13
vacaciones   -> vacacio   IDF = 2.48
mudanza      -> mudanz    IDF = 3.33
MacBook      -> macbook   IDF = 3.33


## Búsqueda

In [4]:
def show(query: str, k: int = 3) -> None:
    print(f"Consulta: {query}")
    for i, score in index.search(query, k=k):
        chunk = chunks[i]
        where = chunk.metadata.get("section", chunk.source)
        print(f"  {score:5.2f}  {chunk.source} | {where}")
    print()


show("¿Cuántos días de vacaciones tengo al año?")
show("dieta diaria viaje España")
show("¿Qué portátil recibe el equipo de diseño?")

Consulta: ¿Cuántos días de vacaciones tengo al año?
   8.02  manual-del-empleado.md | Manual del empleado de NovaTech > Vacaciones
   4.66  beneficios-2026.pdf | beneficios-2026.pdf
   3.39  manual-del-empleado.md | Manual del empleado de NovaTech > Permisos retribuidos

Consulta: dieta diaria viaje España
  15.30  politica-de-gastos-y-viajes.md | Política de gastos y viajes > Dietas
   2.95  politica-de-gastos-y-viajes.md | Política de gastos y viajes > Aprobación de viajes
   2.47  politica-de-gastos-y-viajes.md | Política de gastos y viajes

Consulta: ¿Qué portátil recibe el equipo de diseño?
  13.50  faq-soporte-it.txt | faq-soporte-it.txt
   5.15  faq-soporte-it.txt | faq-soporte-it.txt
   2.76  productos-y-precios.md | Productos y precios de NovaTech > Soporte y SLA



## Efecto de indexar la sección junto al texto

La tabla de permisos no menciona la palabra "permiso". Al indexar cada fragmento con
su sección (`contextualize`), la consulta la encuentra:

In [5]:
plain = BM25Index().fit([c.text for c in chunks])
table = next(i for i, c in enumerate(chunks) if "Mudanza" in c.text)
query = "permisos retribuidos por mudanza"

for name, idx in [("Solo texto", plain), ("Sección + texto", index)]:
    ranking = [i for i, _ in idx.search(query, k=len(chunks))]
    position = ranking.index(table) + 1 if table in ranking else None
    print(f"{name:16} posición de la tabla de permisos: {position}")

Solo texto       posición de la tabla de permisos: 3
Sección + texto  posición de la tabla de permisos: 1


## Limitación: sinónimos y paráfrasis

In [6]:
show("Me han robado el laptop, ¿qué tengo que hacer?")

Consulta: Me han robado el laptop, ¿qué tengo que hacer?



La búsqueda no devuelve ningún resultado: tras la normalización, la consulta solo
conserva los términos "robad" y "laptop", y ninguno aparece en la documentación. Sin
embargo, la FAQ de IT responde exactamente a esta situación ("¿Qué hago si pierdo el
portátil o me lo roban?"). Este problema lo resuelven los embeddings (módulo 04).